# How Large Language Models Work

Large language models (LLMs) are neural networks trained to predict the **next token** in a sequence. Repeating that prediction step produces paragraphs of text, code, or dialogue. At scale, this simple objective captures enough statistical structure in language to support reasoning, summarization, translation, and tool use — when combined with post-training alignment.

This notebook explains LLM mechanics from first principles: the next-token objective, tokenization and embeddings, transformer blocks, attention, causal masking, architecture families, pre-training objectives, scaling behavior, training vs inference, and post-training alignment. Demos use NumPy and Matplotlib so you can reason about logits, softmax, and masking without a GPU.

Prerequisites: **01. Introduction to Generative AI**, **02. Generative AI Models and Probability**, **05. Transformers in Generative AI** (or transformer concepts from **01. AI Foundations → Deep Learning → Transfer Learning and Transformers**).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. The Next-Token Objective

An LLM defines a conditional probability over the vocabulary for each position in a sequence:

$$P(x_{t+1} \mid x_{1:t})$$

Given tokens $x_1, x_2, \ldots, x_t$, the model outputs a distribution over what token comes next. **Generation** repeats this step: sample or select $x_{t+1}$, append it to the context, and predict again until a stop condition.

### 1.1 Autoregressive Generation

```
Context:  "The cat sat on the"
Model:    P(next | context) → "mat" (highest probability)
Append:   "The cat sat on the mat"
Repeat:   P(next | longer context) → ...
```

### 1.2 What the Model Is Not

| Misconception | Reality |
|---------------|--------|
| Lookup table of answers | High-dimensional function mapping context → token distribution |
| Stores facts explicitly | Encodes statistical patterns; facts can be wrong (hallucination) |
| Understands like a human | Predicts plausible continuations; no grounded world model |

Pre-training optimizes **cross-entropy loss** — negative log-likelihood of the true next token — averaged over billions of tokens in web text, books, and code.

---

## 2. From Text to Vectors

Raw text cannot enter a neural network directly. The pipeline has three stages:

```
Text  →  Tokenizer  →  Token IDs  →  Embedding matrix  →  Vectors
                              ↓
                    + Positional encoding
```

| Stage | Role |
|-------|------|
| **Tokenization** | Split text into subword units (BPE, SentencePiece) |
| **Embedding lookup** | Map each token ID to a dense vector $\mathbb{R}^d$ |
| **Positional encoding** | Inject order information — attention alone is permutation-invariant |

Tokenization details — vocabulary size, context limits, cost — are covered in **08. Tokens and Context Windows**. Here we focus on what happens **inside** the model after IDs are embedded.

---

## 3. Transformer Block Anatomy

Modern LLMs stack identical **decoder blocks**. Each block contains:

```
Input vectors
    │
    ▼
┌─────────────────────────┐
│ Masked self-attention   │  ← tokens attend to prior tokens only
└───────────┬─────────────┘
            │ + residual
            ▼
       LayerNorm
            │
            ▼
┌─────────────────────────┐
│ Feed-forward network    │  ← per-token MLP (majority of parameters)
└───────────┬─────────────┘
            │ + residual
            ▼
       LayerNorm
            │
            ▼
Output vectors  →  next block (or final logits)
```

| Component | Purpose |
|-------------|--------|
| **Self-attention** | Each token gathers context from other tokens |
| **FFN** | Non-linear transform at each position; most parameters live here |
| **Residual connections** | Add input to sub-layer output; stabilizes deep training |
| **LayerNorm** | Normalize activations; improves gradient flow |

GPT-style models use **decoder-only** stacks with **causal** (left-to-right) attention. BERT uses **encoder-only** bidirectional attention. T5 and BART use **encoder-decoder** for text-to-text tasks.

---

## 4. Softmax and Next-Token Probabilities

The final layer outputs **logits** $z \in \mathbb{R}^{|V|}$ — one score per vocabulary token. Softmax converts logits to a probability distribution:

$$P_i = \frac{\exp(z_i)}{\sum_j \exp(z_j)}$$

The model is trained to assign high probability to the **correct** next token. At inference, you **sample** or **argmax** from this distribution to produce the next token.

In [ ]:
def softmax(x):
    z = x - np.max(x)
    e = np.exp(z)
    return e / e.sum()

logits = np.array([1.2, 2.6, 1.8, 1.0, 0.9, 2.2])
probs = softmax(logits)

print("Logits → softmax probabilities:")
for i, p in enumerate(probs):
    print(f"  token_{i}: {p:.4f}")
print(f"sum: {probs.sum():.4f}")

---

## 5. Scaled Dot-Product Attention

Attention computes a **weighted sum** of value vectors, where weights come from query–key similarity:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

| Symbol | Meaning |
|--------|--------|
| $Q$ (queries) | "What am I looking for?" — one vector per token |
| $K$ (keys) | "What do I offer?" — one vector per token |
| $V$ (values) | Content to aggregate — one vector per token |
| $d_k$ | Key dimension; $\sqrt{d_k}$ scaling prevents softmax saturation |

**Intuition:** Each token asks (via $Q$) which other tokens (via $K$) are relevant, then blends their content (via $V$). Attention is **dynamic routing** — relevance is computed at runtime, not fixed by position alone.

### 5.1 Multi-Head Attention

Instead of one attention pass, the model runs **multiple heads** in parallel — each with smaller $d_k$, learning different relation patterns (syntax, coreference, local vs long-range). Head outputs are concatenated and projected back to model dimension.

| Design choice | Trade-off |
|---------------|----------|
| More heads | Richer relation types; more compute |
| Fewer, wider heads | Simpler; less parallel specialization |

---

## 6. Causal (Masked) Attention

Decoder-only LLMs must **not** peek at future tokens during training or generation. A **causal mask** sets attention scores to $-\infty$ (or zero after softmax) for positions $j > i$ when computing output at position $i$.

This enforces **autoregressive** consistency: the distribution at step $t$ depends only on $x_{1:t}$.

In [ ]:
n = 12
causal_mask = np.triu(np.ones((n, n), dtype=int), k=1)
allowed = 1 - causal_mask

plt.figure(figsize=(6, 5))
plt.imshow(allowed, cmap="Blues")
plt.title("Causal attention mask (1 = allowed)")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()

Token at row $i$ may attend only to columns $\leq i$. Without this mask, the model could cheat by reading future tokens during training.

---

## 7. Architecture Families

| Architecture | Attention | Typical use | Examples |
|--------------|-----------|-------------|----------|
| **Decoder-only** | Causal (left-to-right) | Text generation, chat | GPT, Llama, Claude |
| **Encoder-only** | Bidirectional | Classification, embeddings | BERT, RoBERTa |
| **Encoder-decoder** | Encoder bidirectional + decoder causal | Text-to-text | T5, BART, original Transformer |

**Rule of thumb:** Match architecture to objective. Generation needs causal decoding; understanding tasks often benefit from bidirectional context; seq2seq tasks use both stacks.

### 7.1 Pre-Training Objectives

| Objective | Full name | How it works | Used by |
|-----------|-----------|--------------|--------|
| **CLM** | Causal language modeling | Predict next token from left context | GPT, Llama |
| **MLM** | Masked language modeling | Predict masked tokens from full context | BERT |

CLM aligns directly with autoregressive generation. MLM learns rich bidirectional representations but requires a separate head for generation.

---

## 8. Greedy Decoding

The simplest inference strategy picks the **highest-probability** token at each step. Greedy decoding is deterministic and fast but can loop or produce bland text.

In [ ]:
vocab = ["language", "model", "attention", "token", "data", "the"]
logits = np.array([1.1, 2.5, 1.9, 0.8, 1.0, 2.2])
probs = softmax(logits)

ranked = sorted(zip(vocab, probs), key=lambda x: x[1], reverse=True)
print("Next-token ranking:")
for token, p in ranked:
    print(f"  {token:12s}  {p:.4f}")
print(f"\nGreedy choice: '{ranked[0][0]}'")

Stochastic sampling (temperature, top-p, top-k) and repetition penalties are covered in **09. Inference Parameters and Text Generation**.

---

## 9. Training at Scale

Pre-training a frontier LLM requires:

| Requirement | Why |
|-------------|-----|
| **Massive text corpora** | Diversity of language, code, reasoning patterns |
| **Distributed GPUs/TPUs** | Single device cannot hold billions of parameters |
| **Mixed precision (fp16/bf16)** | Faster training, lower memory |
| **Learning-rate warmup + decay** | Stable optimization at large batch sizes |
| **Gradient clipping** | Prevent loss spikes from destabilizing training |
| **Checkpointing** | Resume after failures; weeks-long runs |

Data quality matters as much as quantity — deduplication, filtering toxic or low-quality text, and balancing domains (web, books, code) shape downstream behavior.

---

## 10. Scaling Laws

Empirical **scaling laws** show that test loss decreases predictably as **model size**, **dataset size**, and **compute** increase — with diminishing returns:

$$L(N, D) \approx A N^{-\alpha} + B D^{-\beta} + E$$

| Variable | Meaning |
|----------|--------|
| $N$ | Number of parameters |
| $D$ | Training tokens (data scale) |
| $E$ | Irreducible loss floor |

**Emergent capabilities** — tasks that appear suddenly at certain scales — are observed on benchmarks but remain debated; treat them as empirical, not guaranteed.

In [ ]:
train_tokens_b = np.array([1, 5, 20, 80, 300, 1000])
params_b = np.array([0.1, 0.3, 1.3, 7, 34, 175])
synthetic_loss = 3.2 - 0.22 * np.log(train_tokens_b) - 0.18 * np.log(params_b)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(train_tokens_b, synthetic_loss, marker="o")
ax[0].set_xscale("log")
ax[0].set_xlabel("Training tokens (billions)")
ax[0].set_ylabel("Loss (illustrative)")
ax[0].set_title("Loss vs data scale")
ax[1].plot(params_b, synthetic_loss, marker="o", color="darkorange")
ax[1].set_xscale("log")
ax[1].set_xlabel("Parameters (billions)")
ax[1].set_title("Loss vs model scale")
plt.tight_layout()
plt.show()

---

## 11. Parameters, Memory, and Cost

Parameter count drives **capacity**, **VRAM**, and **serving cost**.

| Concern | Larger model | Smaller model |
|---------|--------------|---------------|
| Quality on hard tasks | Usually better | May plateau |
| Inference latency | Higher | Lower |
| Memory (weights only) | More GB | Fewer GB |
| API cost | Higher per token | Lower per token |

Production systems often **route** queries — small model for simple tasks, large model for complex ones.

In [ ]:
models_b = np.array([1, 7, 13, 34, 70, 175])
bytes_per_param_fp16 = 2
memory_gb = models_b * 1e9 * bytes_per_param_fp16 / (1024**3)

print(f"{'Params':>6}  {'Weight memory (fp16)':>22}")
print("-" * 32)
for m, mem in zip(models_b, memory_gb):
    print(f"{int(m):>4}B  {mem:>18.2f} GB")
print("\nNote: inference also needs KV-cache and activations — often 2–3× weight memory.")

---

## 12. Training vs Inference

| Phase | What happens | Compute profile |
|-------|----------------|-----------------|
| **Training** | Forward + backward pass; update weights | Very heavy; weeks on clusters |
| **Inference** | Forward pass only; generate tokens | Lighter per step; scales with users × output length |

Decoding strategy affects inference:

| Strategy | Behavior |
|----------|----------|
| **Greedy** | Argmax each step; deterministic |
| **Sampling** | Random draw from distribution; diverse |
| **Beam search** | Keep top-$k$ partial sequences; used in translation |

Each generated token requires a full forward pass through the model — long outputs are expensive.

---

## 13. Post-Training and Alignment

Pre-training teaches **language** — not necessarily **helpful assistant behavior**. Post-training stages adapt the base model:

```
Base model (pre-trained on web text)
        │
        ▼
Supervised fine-tuning (SFT)     ← instruction-response pairs
        │
        ▼
Preference optimization          ← RLHF, DPO — helpful vs harmful
        │
        ▼
Deployed assistant
```

Alignment **steers** behavior on top of pre-trained capabilities — it does not replace core language modeling. Details: **11. Introduction to Fine-Tuning and Alignment**.

---

## 14. Failure Modes and Limitations

| Failure | Description | Mitigation |
|---------|-------------|------------|
| **Hallucination** | Confident but false statements | RAG, citations, eval, lower temperature |
| **Prompt sensitivity** | Small wording changes alter output | System prompts, few-shot examples |
| **Context limits** | Cannot see beyond context window | Chunking, summarization, retrieval |
| **Stale knowledge** | Training cutoff date | RAG, tool use, web search |
| **Unsafe outputs** | Harmful or biased content | Alignment, guardrails, moderation |

Benchmark scores alone do not guarantee production readiness — evaluate on **your** tasks and monitor in deployment.

---

## 15. Deployment Trade-offs

Choosing and operating an LLM balances four constraints:

```
        Quality
           ▲
           │
  Safety ◄─┼─► Latency
           │
           ▼
          Cost
```

| Decision | Consider |
|----------|----------|
| Model size | Quality vs latency/cost |
| Hosted API vs self-host | Ops burden vs data control |
| Context length | Long docs vs memory cost |
| Fallback strategy | Smaller model or cached response when primary fails |

Model family comparison: **10. Model Families and Choosing an LLM**. Inference tuning: **09. Inference Parameters and Text Generation**.

---

## 16. Common Mistakes

| Mistake | Why it matters |
|---------|----------------|
| Treating LLM output as ground truth | Models predict plausible text, not verified facts |
| Ignoring token/context limits | Requests fail or truncate silently |
| Using encoder-only models for generation | BERT cannot autoregress without extra machinery |
| Skipping alignment for user-facing apps | Base models follow text, not instructions |
| One model size for all traffic | Over-provisioning wastes cost; under-provisioning hurts quality |

---

## 17. Summary

LLMs are **autoregressive next-token predictors** built from stacked **transformer decoder blocks** with **causal self-attention**, **FFNs**, **residual connections**, and **LayerNorm**. Pre-training on massive corpora captures language structure; **post-training** aligns behavior for assistants.

Performance scales with parameters, data, and compute (**scaling laws**), but deployment requires balancing quality, latency, cost, and safety.

Next in this section:

- **08. Tokens and Context Windows** — tokenization, context limits, cost
- **10. Model Families and Choosing an LLM** — GPT, Claude, Llama, deployment choices
- **09. Inference Parameters and Text Generation** — temperature, top-p, decoding
- **11. Introduction to Fine-Tuning and Alignment** — conceptual overview (SFT, RLHF, DPO)
- **09. Open Source Models → 07. Fine-Tuning Open Models** — hands-on LoRA/QLoRA/TRL
- **10. Production Fine-Tuning and Alignment** — cloud fine-tuning and alignment depth